1.Data Ingestion

In [0]:

# 1. Spark Session

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ChicagoCrime_BigData_Project") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

spark

In [0]:

# 2. Define Schema

from pyspark.sql.types import *

crime_schema = StructType([
    StructField("ID", LongType(), True),
    StructField("Case_Number", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Block", StringType(), True),
    StructField("IUCR", StringType(), True),
    StructField("Primary_Type", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Location_Description", StringType(), True),
    StructField("Arrest", BooleanType(), True),
    StructField("Domestic", BooleanType(), True),
    StructField("Beat", IntegerType(), True),
    StructField("District", IntegerType(), True),
    StructField("Ward", IntegerType(), True),
    StructField("Community_Area", IntegerType(), True),
    StructField("FBI_Code", StringType(), True),
    StructField("X_Coordinate", DoubleType(), True),
    StructField("Y_Coordinate", DoubleType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Updated_On", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True)
])

In [0]:

# 3. Load CSV

crime_raw_df = spark.read \
    .option("header", "true") \
    .schema(crime_schema) \
    .csv("dbfs:/Volumes/workspace/crime_data/crime/Crimes_-_2001_to_Present_20260226.csv")

crime_raw_df.printSchema()
crime_raw_df.count()

root
 |-- ID: long (nullable = true)
 |-- Case_Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location_Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- FBI_Code: string (nullable = true)
 |-- X_Coordinate: double (nullable = true)
 |-- Y_Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated_On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



8502217

In [0]:

# 4. Basic Data Validation

from pyspark.sql.functions import col, count, when

crime_raw_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in crime_raw_df.columns
]).show()

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
| ID|Case_Number|Date|Block|IUCR|Primary_Type|Description|Location_Description|Arrest|Domestic|Beat|District|  Ward|Community_Area|FBI_Code|X_Coordinate|Y_Coordinate|Year|Updated_On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|  0|          0|   0|    0|   0|           0|          0|               15642|     0|       0|   0|      47|614818|        613685|       0|       94705|       94705|   0|         0|   94705|    94705|   94705|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+---

In [0]:

# 5. Cleaning

crime_clean_df = crime_raw_df.dropna(
    subset=["Arrest", "Primary_Type", "District", "Year"]
)

crime_clean_df = crime_clean_df.dropDuplicates(["ID"])
crime_clean_df.count()

8502170

In [0]:

# 6. Rename Columns

crime_df_clean = crime_clean_df

for column in crime_df_clean.columns:
    new_column = column.strip() \
                       .replace(" ", "_") \
                       .replace("-", "_") \
                       .replace("/", "_") \
                       .replace(".", "_")
    crime_df_clean = crime_df_clean.withColumnRenamed(column, new_column)

crime_df_clean.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Case_Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location_Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- FBI_Code: string (nullable = true)
 |-- X_Coordinate: double (nullable = true)
 |-- Y_Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated_On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA crime_data")

spark.sql("DROP TABLE IF EXISTS crime_parquet_table")

DataFrame[]

In [0]:
crime_df_clean.write \
    .mode("overwrite") \
    .partitionBy("Year") \
    .saveAsTable("crime_parquet_table")

print("Parquet table created successfully.")

Parquet table created successfully.
